In [1]:
# --- Setup: quiet logs/prints and bypass Redis for notebook testing ---

import io, sys, logging, warnings
from contextlib import contextmanager
from typing import Optional, Dict, Any, List
import pandas as pd

# 1) Quiet warnings & common noisy loggers
warnings.filterwarnings("ignore")
for name in [
    "azure.core.pipeline",
    "azure.core.pipeline.policies.http_logging_policy",
    "azure.ai.inference",
    "httpx",
    "urllib3",
    # your modules:
    "src.agents.router_agent",
    "src.agents.generic_agent",
    "src.agents.conversationalist_agent",
    "src.agents.specialized_sql_layer",
    "src.agents.openai_agent",
    "src.agents.parse_intent_agent",
    "src.agents.building_flow_graph",
    "src.database.vector_client",
]:
    logging.getLogger(name).setLevel(logging.ERROR)
logging.getLogger().setLevel(logging.ERROR)

# 2) Simple stdout/err suppressor to hide print() noise (e.g., VectorClient banners)
@contextmanager
def suppress_output(enabled: bool = True):
    if not enabled:
        yield
        return
    old_out, old_err = sys.stdout, sys.stderr
    try:
        sys.stdout, sys.stderr = io.StringIO(), io.StringIO()
        yield
    finally:
        sys.stdout, sys.stderr = old_out, old_err

# 3) Bypass Redis: monkeypatch BuildingAgent session store to in-memory
import src.agents.building_agent as building_agent_mod
_INMEM_SESS: Dict[str, Any] = {}
def _fake_get_session_state(thread_id): return _INMEM_SESS.setdefault(thread_id, {"history": []})
def _fake_update_session_state(thread_id, new_state): _INMEM_SESS[thread_id] = new_state or {}
building_agent_mod.get_session_state = _fake_get_session_state
building_agent_mod.update_session_state = _fake_update_session_state

# 4) Quietly initialize the AgentRouter once and reuse it
from src.pipeline.agent_router import AgentRouter

_ROUTER: Optional[AgentRouter] = None
def get_router() -> AgentRouter:
    global _ROUTER
    if _ROUTER is None:
        with suppress_output(True):
            _ROUTER = AgentRouter()
    return _ROUTER

def _short(s: Optional[str], n: int = 160) -> str:
    if not s: return "—"
    s = str(s).replace("\n", " ").strip()
    return s if len(s) <= n else s[:n] + "…"


Connection to Vector Database established.
Connection to Vector Database established.
✅ VectorClient initialized and connected.


### The following code block runs test cases

In [2]:
# --- AgentRouter scenario tester (classification + intent) ---

from typing import List

router = get_router()

SCENARIOS: List[Dict[str, Any]] = [
    # Conversational
    {"name": "CONV: greeting", "question": "hey there, how are you?", "metadata": {},
     "expected_classification": "conversational", "expected_intent": None},
    {"name": "CONV: small talk", "question": "tell me a fun fact about energy", "metadata": {},
     "expected_classification": "conversational", "expected_intent": None},

    # Generic (not building-specific)
    {"name": "GEN: general efficiency tips", "question": "Give me general tips to save energy at home.",
     "metadata": {}, "expected_classification": "generic", "expected_intent": None},
    {"name": "GEN: define EPC", "question": "What does EPC mean in buildings?",
     "metadata": {}, "expected_classification": "generic", "expected_intent": None},

    # Building-specific → query generic database
    {"name": "BLD-GENSQL: energy rating (Hammarby address in text)",
     "question": "I live at Hammarby Allé 24, what’s my energy class?",
     "metadata": {}, "expected_classification": "building_specific", "expected_intent": "query generic database"},
    {"name": "BLD-GENSQL: heated area (address in metadata)",
     "question": "What is the heated area (Atemp)?",
     "metadata": {"address": "Exempelgatan 12"}, "expected_classification": "building_specific",
     "expected_intent": "query generic database"},

    # Building-specific → query specific database (specialized SQL / detailed tables)
    {"name": "BLD-SPECSQL: HVAC types",
     "question": "For Exempelgatan 12, list the HVAC system type(s) and ventilation system.",
     "metadata": {"address": "Exempelgatan 12"}, "expected_classification": "building_specific",
     "expected_intent": "query specific database"},
    {"name": "BLD-SPECSQL: end-use breakdown",
     "question": "Show electricity end-use breakdown for Exempelgatan 12.",
     "metadata": {"address": "Exempelgatan 12"}, "expected_classification": "building_specific",
     "expected_intent": "query specific database"},

    # Building-specific → query vector database (advice)
    {"name": "BLD-VECTOR: improve insulation",
     "question": "I live at Exempelgatan 12. How can I improve insulation?",
     "metadata": {"address": "Exempelgatan 12"}, "expected_classification": "building_specific",
     "expected_intent": "query vector database"},
    {"name": "BLD-VECTOR: reduce CO2",
     "question": "How can I reduce my building’s CO2 footprint over the next year?",
     "metadata": {"address": "Exempelgatan 12"}, "expected_classification": "building_specific",
     "expected_intent": "query vector database"},

    # Building-specific → simulations (what-if)
    {"name": "BLD-SIM: solar CO2 reduction",
     "question": "If we add solar panels at Exempelgatan 12, how much CO2 do we save?",
     "metadata": {"address": "Exempelgatan 12"}, "expected_classification": "building_specific",
     "expected_intent": "simulations"},
    {"name": "BLD-SIM: insulation savings",
     "question": "How much energy would we save if we upgrade insulation?",
     "metadata": {"address": "Exempelgatan 12"}, "expected_classification": "building_specific",
     "expected_intent": "simulations"},

    # Extra generic vs building mix
    {"name": "GEN: ventilation best practices",
     "question": "What are ventilation best practices to balance air quality and energy?",
     "metadata": {}, "expected_classification": "generic", "expected_intent": None},
    {"name": "BLD-GENSQL: electricity kWh",
     "question": "For Exempelgatan 12, how many kWh of electricity do we use annually?",
     "metadata": {"address": "Exempelgatan 12"}, "expected_classification": "building_specific",
     "expected_intent": "query generic database"},
]

def run_suite(cases: List[Dict[str, Any]], thread_id: str = "notebook:test-session") -> pd.DataFrame:
    rows = []
    for case in cases:
        q = case["question"]
        md = case.get("metadata") or {}
        msgs = [{"role": "user", "content": q}]
        with suppress_output(True):
            out = router.route_message(messages=msgs, last_message=q, metadata=md, thread_id=thread_id)

        rows.append({
            "name": case["name"],
            "question": q,
            "expected_classification": case.get("expected_classification"),
            "actual_classification": out.get("classification"),
            "expected_intent": case.get("expected_intent"),
            "actual_intent": out.get("intent"),
            "agent_answered": out.get("agent_answered"),
            "content_preview": _short(out.get("content")),
        })
    df = pd.DataFrame(rows)

    # Accuracy summaries
    df["classification_correct"] = (df["expected_classification"].fillna("—") == df["actual_classification"].fillna("—"))
    df["intent_correct"] = (df["expected_intent"].fillna("—") == df["actual_intent"].fillna("—"))

    # Overall metrics
    overall_class = round(df["classification_correct"].mean() * 100, 1)
    bld_mask = (df["expected_classification"] == "building_specific")
    overall_intent_bld = round(df.loc[bld_mask, "intent_correct"].mean() * 100, 1) if bld_mask.any() else None

    print("=== Summary ===")
    print(f"Classification accuracy (all): {overall_class}%")
    if overall_intent_bld is not None:
        print(f"Intent accuracy (building_specific rows): {overall_intent_bld}%")
    else:
        print("No building_specific rows to compute intent accuracy.")

    return df.sort_values(by=["expected_classification", "name"]).reset_index(drop=True)

df_results = run_suite(SCENARIOS)
display(df_results)


INFO:openai._base_client:Retrying request to /chat/completions in 4.000000 seconds
INFO:openai._base_client:Retrying request to /chat/completions in 60.000000 seconds
INFO:openai._base_client:Retrying request to /chat/completions in 60.000000 seconds
ERROR:src.agents.specialized_sql_layer:Azure OpenAI error: Error code: 429 - {'error': {'code': '429', 'message': 'Requests to the ChatCompletions_Create Operation under Azure OpenAI API version 2025-01-01-preview have exceeded token rate limit of your current OpenAI S0 pricing tier. Please retry after 60 seconds. Please go here: https://aka.ms/oai/quotaincrease if you would like to further increase the default rate limit. For Free Account customers, upgrade to Pay as you Go here: https://aka.ms/429TrialUpgrade.'}}
Traceback (most recent call last):
  File "c:\Users\shada\Documents\spara-backend\src\agents\specialized_sql_layer.py", line 214, in execute
    plan = self._synthesize_sql_plan(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\

=== Summary ===
Classification accuracy (all): 92.9%
Intent accuracy (building_specific rows): 88.9%


,name,question,expected_classification,actual_classification,expected_intent,actual_intent,agent_answered,content_preview,classification_correct,intent_correct
0,BLD-GENSQL: electricity kWh,"For Exempelgatan 12, how many kWh of electrici...",building_specific,building_specific,query generic database,query generic database,sql,No results to summarize.,True,True
1,BLD-GENSQL: energy rating (Hammarby address in...,"I live at Hammarby Allé 24, what’s my energy c...",building_specific,building_specific,query generic database,query generic database,unknown,Can you please provide the building address?,True,True
2,BLD-GENSQL: heated area (address in metadata),What is the heated area (Atemp)?,building_specific,generic,query generic database,None,generic,**Heated Area (Atemp) Definition:** - **Atemp...,False,False
3,BLD-SIM: insulation savings,How much energy would we save if we upgrade in...,building_specific,building_specific,simulations,simulations,simulation,Summary of: Simulation result,True,True
4,BLD-SIM: solar CO2 reduction,"If we add solar panels at Exempelgatan 12, how...",building_specific,building_specific,simulations,simulations,simulation,Summary of: Simulation result,True,True
5,BLD-SPECSQL: HVAC types,"For Exempelgatan 12, list the HVAC system type...",building_specific,building_specific,query specific database,query specific database,unknown,Specialized SQL error = llm_error: Error code:...,True,True
6,BLD-SPECSQL: end-use breakdown,Show electricity end-use breakdown for Exempel...,building_specific,building_specific,query specific database,query specific database,unknown,Specialized SQL error = llm_error: Error code:...,True,True
7,BLD-VECTOR: improve insulation,I live at Exempelgatan 12. How can I improve i...,building_specific,building_specific,query vector database,query vector database,vector,No content returned from the model.,True,True
8,BLD-VECTOR: reduce CO2,How can I reduce my building’s CO2 footprint o...,building_specific,building_specific,query vector database,query vector database,vector,No content returned from the model.,True,True
9,CONV: greeting,"hey there, how are you?",conversational,conversational,None,None,conversationalist,"Hi there! I'm just a system, but I'm here and ...",True,True


### The following code block runs individual cases

In [3]:
def ask(question: str, *, metadata: Optional[Dict[str, Any]] = None,
        thread_id: str = "notebook:adhoc") -> pd.DataFrame:
    """
    Single question quick test. Returns a 1-row DataFrame with
    classification, intent, agent answer label, and a short preview.
    """
    r = get_router()
    msgs = [{"role": "user", "content": question}]
    with suppress_output(True):
        out = r.route_message(messages=msgs, last_message=question, metadata=(metadata or {}), thread_id=thread_id)
    row = {
        "question": question,
        "classification": out.get("classification"),
        "intent": out.get("intent"),
        "agent_answered": out.get("agent_answered"),
        "content_preview": _short(out.get("content")),
    }
    return pd.DataFrame([row])

In [4]:
# Conversational
display(ask("hey there, how are you?"))

# Generic (not building-specific)
display(ask("What does EPC mean in buildings?"))


,question,classification,intent,agent_answered,content_preview
0,"hey there, how are you?",conversational,None,conversationalist,"Hi there! I'm just a system, but I'm here and ..."


,question,classification,intent,agent_answered,content_preview
0,What does EPC mean in buildings?,generic,None,generic,"EPC stands for ""Energiprestanda­certifikat"" in..."


In [5]:
# Address inside the question → building_specific
display(ask("I live at Hammarby Allé 24, what’s my energy class?"))


,question,classification,intent,agent_answered,content_preview
0,"I live at Hammarby Allé 24, what’s my energy c...",building_specific,query generic database,unknown,Can you please provide the building address?


In [6]:
# Provide the address as metadata when it's not in the question
display(ask("What is the heated area (Atemp)?", metadata={"address": "Exempelgatan 12"}))


,question,classification,intent,agent_answered,content_preview
0,What is the heated area (Atemp)?,generic,None,generic,**Heated Area (Atemp) Definition:** - **Atemp...
